# Stage 6M: Multi-Model Cumulative Adaptation & Confirmatory Benchmark
## Official Cloud GPU Execution Notebook (Kaggle NVIDIA Tesla T4 × 2)

**Governing Protocol:** Stage 6M v1.5 Five-Stratum Benchmark (Section 14.7)  
**Governing Branch:** `stage6m-five-stratum-parity` @ `baee3b8381389ffde38e907dc137623748c254a8`  
**Restricted Online Manifest:** `datasets/splits/stage6m_cohort3_online_inference_manifest.json`  
**Expected Online Manifest SHA-256:** `0f1cca979a0b7c9872ce576df97a3db48c84358db3b4f287aa77d76a779751a1`  
**Parent Locked Manifest SHA-256:** `171815faab962073b7d446bc6d755602c948386f62437876b5fe54fb82b4f3d0`  
**Target Accelerator:** NVIDIA Tesla T4 × 2 (Dual 16 GB VRAM GPUs; Tesla P100 is retired)  

---
### Three-Arm Matched Experimental Design
1. **Arm 1 (Static Baseline):** Fixed unadapted reference ($\\theta_0$) across 4,125 sequential stream windows.
2. **Arm 2 (Ungated CTTA):** Continuous test-time adaptation via SUTA (LayerNorm affine updates).
3. **Arm 3 (DSG Gate-Controlled CTTA):** Continuous adaptation governed by Disparity Safety Gate on Sentinel Panel with Lan-DeMets O'Brien-Fleming alpha spending and atomic candidate live rollback on rejection.

**Dataset & Transcript Shielding Accounting:**
- **Evaluation Cohort:** 1,100 speakers / 16,500 clips (220 per stratum across Australia, Canada, England, South Asian, US).
- **Evaluation Reference Labels:** **STRICTLY SEALED** (`transcript == null` in online manifest).
- **Gate-Reference Panel:** 100 speakers / 1,500 clips (20 per stratum, labels active for live Sentinel WER calculation).
- **Panel Overlap:** 0 (strictly disjoint).

In [ ]:
# Step 1: Hardware Accelerator Verification (NVIDIA T4 x2)
import sys, platform, torch

print('=' * 80)
print('STAGE 6M KAGGLE EXECUTION: HARDWARE ACCELERATOR CHECK')
print('=' * 80)
cuda_avail = torch.cuda.is_available()
print(f'CUDA Available: {cuda_avail}')
if cuda_avail:
    dev_count = torch.cuda.device_count()
    dev_name_0 = torch.cuda.get_device_name(0)
    vram_gb_0 = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'Device Count:     {dev_count}')
    print(f'Primary GPU (0):  {dev_name_0} ({vram_gb_0:.2f} GB VRAM)')
    if dev_count > 1:
        dev_name_1 = torch.cuda.get_device_name(1)
        vram_gb_1 = torch.cuda.get_device_properties(1).total_memory / (1024 ** 3)
        print(f'Secondary GPU (1): {dev_name_1} ({vram_gb_1:.2f} GB VRAM) [NVIDIA T4 x2 Dual Topology Confirmed]')
    assert vram_gb_0 >= 14.0, f'VRAM {vram_gb_0:.2f} GB below expected 16GB profile!'
    print('STATUS: GPU accelerator confirmed ready.')
else:
    print('[ERROR] No GPU detected! Please configure Notebook Settings -> Accelerator -> GPU T4 x2.')


In [ ]:
# Step 2: Workspace Setup & Exact Dependency Installation
import os, subprocess

if not os.path.exists('datasets') and os.path.exists('/kaggle/input/stage6m-frozen-benchmark'):
    print('Kaggle frozen benchmark dataset mounted at /kaggle/input/stage6m-frozen-benchmark/')

# Install complementary metrics and audio libraries (soundfile 0.14.0 is preinstalled)
!pip install -q editdistance==0.8.1 statsmodels==0.14.2 miniaudio==1.59
print('Dependencies verified successfully.')


In [ ]:
# Step 3: Cryptographic Lock & Transcript Shielding Verification
import hashlib
from pathlib import Path

possible_paths = [
    Path('/kaggle/input/stage6m-frozen-benchmark/splits/stage6m_cohort3_online_inference_manifest.json'),
    Path('/kaggle/input/stage6m-frozen-benchmark/stage6m_cohort3_online_inference_manifest.json'),
    Path('datasets/splits/stage6m_cohort3_online_inference_manifest.json'),
    Path('/kaggle/input/stage6m-frozen-benchmark/splits/stage6m_cohort3_final_manifest.json'),
    Path('/kaggle/input/stage6m-frozen-benchmark/stage6m_cohort3_final_manifest.json'),
    Path('datasets/splits/stage6m_cohort3_final_manifest.json')
]

target_manifest = None
for p in possible_paths:
    if p.exists():
        target_manifest = p
        break

assert target_manifest is not None, 'Could not locate official manifest!'
manifest_sha256 = hashlib.sha256(target_manifest.read_bytes()).hexdigest()
expected_online = '0f1cca979a0b7c9872ce576df97a3db48c84358db3b4f287aa77d76a779751a1'
expected_parent = '171815faab962073b7d446bc6d755602c948386f62437876b5fe54fb82b4f3d0'

print(f'Manifest Path:   {target_manifest}')
print(f'Manifest Size:   {target_manifest.stat().st_size:,} bytes')
print(f'Observed SHA256: {manifest_sha256}')
if manifest_sha256 == expected_online:
    print('CRITICAL CHECK PASSED: Restricted online inference manifest authentic (transcripts shielded).')
elif manifest_sha256 == expected_parent:
    print('CRITICAL CHECK PASSED: Parent locked manifest authentic (runner will enforce in-memory shielding).')
else:
    raise ValueError(f'Digest mismatch! Unexpected hash {manifest_sha256}')


In [ ]:
# Step 4: Launch Confirmatory 3-Arm Benchmark
# Executes the three-arm stream across all 4,125 streaming windows
!python cloud/kaggle/run_stage6m_cohort3_kaggle_benchmark.py


In [ ]:
# Step 5: Bundle Output Results for Download
!tar -czvf /kaggle/working/stage6m_cohort3_results.tar.gz -C /kaggle/working stage6m_cohort3_results checkpoints
import os
if os.path.exists('/kaggle/working/stage6m_cohort3_results.tar.gz'):
    sz = os.path.getsize('/kaggle/working/stage6m_cohort3_results.tar.gz')
    print(f'Results packaged successfully: /kaggle/working/stage6m_cohort3_results.tar.gz ({sz:,} bytes)')
    print('Download this file from the Kaggle Output pane on the right.')
